In [ ]:
import torch
import torch.nn as nn
import torchinfo
from result.repondeur import prediction_to_csv
import torchvision
import numpy as np
import csv
from tqdm import tqdm
from model.loader import load_dataset, load_test_loader

import torchvision.models as models


In [ ]:
# Check for GPU availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✓ Using GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("✓ Using Apple Silicon GPU (MPS)")
else:
    device = torch.device("cpu")
    print("⚠ Using CPU - training will be slow!")

In [ ]:
def train_crossEntropyLoss(train_set, model, epochs):
    """
    Entraine un model à partir d'un train_set donné. Retourne les metrics de l'entrainement
    """

    # Parameters
    model.train()
    
    criterion = nn.CrossEntropyLoss()
    
    optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)

    # optimizer = torch.optim.SGD(
    #     model.parameters(),
    #     lr=0.1,
    #     momentum=0.9,
    #     weight_decay=5e-4
    # )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=200
    )


    accuracies = np.zeros(epochs)
    losses = np.zeros(epochs)

    for epoch in range(epochs):
        total_loss = 0.0
        correct = 0
        total = 0

        pbar = tqdm(
            train_set,
            desc=f"Model training | Epoch {epoch+1}/{epochs}",
            leave=True
        )

        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)

            #prediction
            logits = model(images)
            loss = criterion(logits, labels)

            #propagation du gradient
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Calcul des metriques
            total_loss += loss.item()
            
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            pbar.set_postfix({
                "batch_loss": f"{loss.item():.4f}",
                "accuracy": f"{correct/total*100:.2f}%"
            })
        accuracies[epoch] = correct/total
        losses[epoch] = total_loss

        scheduler.step()
    return accuracies, losses

In [ ]:
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from utils.CustomDataset import CustomDataset
from utils.CustomSkibidiDataset import CustomSkibidiDataset

from torchvision.transforms import v2

In [ ]:
# 1. Define Image Transforms
# HUUUUGOOOOOOOO
# transform = transforms.Compose([
#     transforms.Resize((224, 224)),
#     torchvision.transforms.functional.to_dtype(float16, scale=True),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])

transform = v2.Compose([
    #v2.ToImage(),                  # Convert to Tensor (if it's PIL)
    v2.Resize((224, 224)),
    v2.ToDtype(torch.float32, scale=True), # Scales to float16 [0, 1] # TODO
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


# 2. Instantiate your Dataset
"""train_dataset = CustomDataset(
    img_dir='data/',
    annotations_file='data/train.csv',
    transform=transform, 
    target_transform=None # ???
)"""

skibidi_dataset = CustomSkibidiDataset(
    img_dir='data/treated_train/',
    csv_mapper='data/treated_train.csv',
    transform=transform, 
    target_transform=None # ???
)

train_size = int(0.8 * len(skibidi_dataset))
test_size = len(skibidi_dataset) - train_size

train_dataset, validation_dataset = random_split(skibidi_dataset, [train_size, test_size])

In [ ]:
epochs = 20
model = models.resnet50(pretrained=True)
for param in model.parameters():
    param.requires_grad = False
num_classes = 50
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)

model.to(device)
train_set = DataLoader(
    dataset=train_dataset,
    batch_size=32,      # Number of images per batch
    shuffle=True,       # Shuffle every epoch to prevent overfitting
    num_workers=4,      # Number of CPU cores for data loading
    pin_memory=True     # Speeds up transfer to GPU
)

accuracies, losses = train_crossEntropyLoss(train_set,model, epochs)
try : 
    np.save("./ACCURACIES.npy", accuracies)
    np.save("./LOSSES.npy", losses)
except :
    pass
torch.save(model.state_dict(), f"model/trained_model/MiniCNN_augmented_data_{epochs}_epochs.pth")

In [28]:
import pandas as pd
from eval.Analyser import Analyser

In [29]:
data = pd.read_csv("prediction.csv")

analyser = Analyser(data)
report = analyser.generate_report()
print(report)

{'f1_score_avg': 0.28708540208615774, 'f1_score_per_class': array([0.30769231, 0.0483871 , 0.72340426, 0.32542373, 0.02352941,
       0.05405405, 0.64761905, 0.61654135, 0.28865979, 0.14285714,
       0.25930101, 0.        , 0.        , 0.22222222, 0.32490975,
       0.53623188, 0.16748768, 0.2893617 , 0.06802721, 0.51282051,
       0.4375    , 0.07246377, 0.52475248, 0.17748344, 0.29559748,
       0.19444444, 0.        , 0.61702128, 0.27906977, 0.58823529,
       0.05333333, 0.15873016, 0.0754717 , 0.28169014, 0.10891089,
       0.6342711 , 0.29565217, 0.29239766, 0.26515152, 0.08484848,
       0.28888889, 0.30120482, 0.53676471, 0.04968944, 0.22754491,
       0.31683168, 0.2195122 ]), 'best_f1': np.int64(2)}
